# SmartPark AI Week 3 FYP Data Pipeline

This notebook ingests and prepares the miniature parking data for Topic 4. It uses reviewed detector-pilot images and consecutive indexed replay frames from five FYP recordings. The original camera imagery is in the approved private input bundle. The repository carries the pipeline, portable image index, data fingerprints and execution evidence.

Set SMARTPARK_FYP_INPUT_ZIP to the path of smartpark_fyp_review_input.zip, or use the upload prompt in an approved Colab runtime. Read the Week 3 inventory and pipeline decisions for source, access and scope. No model key is required.

## Part A Source and requirements

| Topic 2 requirement | Needed input | Selected source |
|---|---|---|
| FR1 and RAI1 Occupancy and safe guidance | Visible toy cars, bay coverage and explicit unknown states | Detector-pilot JPEGs and indexed camera replay with bay-state logs |
| FR2 and RAI2 Tracking and uncertainty | Consecutive frames joined to session-scoped track IDs and alignment status | Five FYP replay sessions with aligned indices, timestamps and track observations |

In [1]:
from collections import Counter
from hashlib import sha256
from io import BytesIO
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile
import csv
import importlib.util
import json
import os

from PIL import Image

COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if "SMARTPARK_FYP_INPUT_ZIP" in os.environ:
    input_zip = Path(os.environ["SMARTPARK_FYP_INPUT_ZIP"])
elif COLAB and Path("/content/smartpark_fyp_review_input.zip").is_file():
    input_zip = Path("/content/smartpark_fyp_review_input.zip")
elif COLAB:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose the single private FYP input ZIP.")
    input_zip = BytesIO(next(iter(uploaded.values())))
else:
    raise ValueError("Set SMARTPARK_FYP_INPUT_ZIP to the private input ZIP.")

output = Path(os.environ.get("SMARTPARK_WEEK3_OUTPUT_DIR", "smartpark_fyp_week3"))
data = output / "data"
processed = data / "processed"
pipeline = data / "pipeline_v1/images"
processed.mkdir(parents=True, exist_ok=True)
pipeline.mkdir(parents=True, exist_ok=True)

with ZipFile(input_zip) as archive:
    raw = archive.read("manifest.json")
    source_manifest = json.loads(raw)
    archive_entries = Counter(archive.namelist())
manifest_sha256 = sha256(raw).hexdigest()
source_rows = source_manifest["records"]
assert source_manifest["format"] == 1 and source_rows
assert len({row["record_id"] for row in source_rows}) == len(source_rows)
assert len({row["image_path"] for row in source_rows}) == len(source_rows)
kind_counts = Counter(row["kind"] for row in source_rows)
print(f"Ingested {len(source_rows)} candidates. Manifest SHA-256 {manifest_sha256}.")
print(dict(kind_counts))

Ingested 285 candidates. Manifest SHA-256 11b270f4d4fb7c97cb81463505881cab037f497c514ad2a2f0e2a3af95d83612.
{'detector_image': 85, 'annotated_replay': 200}


## Part B Inspect, compare and prepare

Inspect one detector image and one replay frame to verify how the formats and labels differ. Compare the original geometry with an aspect-ratio-preserving preview at up to 640 pixels per side. The pipeline keeps the original JPEG bytes for both roles. This preserves small roof-tag detail, the frames' displayed geometry and their existing annotation links. It adds type-checked metadata and portable source references without applying a new image transform.

In [2]:
def compare_geometry(archive, role):
    row = next(entry for entry in source_rows if entry["kind"] == role)
    with Image.open(BytesIO(archive.read(row["image_path"]))) as image:
        image.load()
        preview = image.copy()
        preview.thumbnail((640, 640), Image.Resampling.LANCZOS)
        return {
            "role": role, "format": image.format, "mode": image.mode,
            "native_size": image.size, "resized_preview_size": preview.size,
        }

with ZipFile(input_zip) as archive:
    for role in ("detector_image", "annotated_replay"):
        print(compare_geometry(archive, role))

{'role': 'detector_image', 'format': 'JPEG', 'mode': 'RGB', 'native_size': (1040, 1015), 'resized_preview_size': (640, 625)}
{'role': 'annotated_replay', 'format': 'JPEG', 'mode': 'RGB', 'native_size': (1920, 1080), 'resized_preview_size': (640, 360)}


In [3]:
fields = [
    "record_id", "kind", "source_id", "image_path", "sha256", "width", "height", "mode",
    "capture_date", "frame", "seconds", "window", "box_count", "bay_count", "track_count",
    "annotation_origin",
]

def prepare_index(archive):
    rows = []
    for item in source_rows:
        path = item["image_path"]
        parts = PurePosixPath(path).parts
        if not parts or parts[0] != "images" or ".." in parts or archive_entries[path] != 1:
            raise ValueError(f"Invalid or missing image reference: {item['record_id']}")
        payload = archive.read(path)
        fingerprint = sha256(payload).hexdigest()
        if fingerprint != item["sha256"]:
            raise ValueError(f"Source fingerprint mismatch: {item['record_id']}")
        with Image.open(BytesIO(payload)) as image:
            if image.format != "JPEG" or image.mode != "RGB":
                raise ValueError(f"Expected an RGB JPEG: {item['record_id']}")
            image.verify()
        with Image.open(BytesIO(payload)) as image:
            image.load()
            width, height = image.size
        row = {k: item.get(k, "") for k in fields}
        row.update(width=width, height=height, mode="RGB")
        if item["kind"] == "detector_image":
            if len(item["boxes"]) != item["box_count"]:
                raise ValueError(f"Detached detector labels: {item['record_id']}")
        elif item["kind"] == "annotated_replay":
            row["bay_count"] = len(item["bay_states"])
            row["track_count"] = len(item["tracks"])
        else:
            raise ValueError(f"Unknown source role: {item['kind']}")
        rows.append(row)
    return rows

with ZipFile(input_zip) as archive:
    prepared = prepare_index(archive)
assert len(prepared) == len(source_rows)
index_path = processed / "image_manifest.csv"
with index_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(prepared)
print(f"Prepared {len(prepared)} image references with readable RGB JPEGs and verified source hashes.")

Prepared 285 image references with readable RGB JPEGs and verified source hashes.


## Part C Execute, verify and register the V1 pipeline

Rerun the pipeline on the same input and compare the ordered metadata rows and image fingerprints. The index carries the portable references. The lineage manifest pins the private input. The run manifest records source coverage, image geometry and the verified hand-off. Topic 4 uses the same private manifest for its detailed metadata review.

In [4]:
with ZipFile(input_zip) as archive:
    second_run = prepare_index(archive)
if second_run != prepared:
    raise RuntimeError("Week 3 repeatability check failed")

def write_json(path, content):
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(json.dumps(content, indent=2) + "\n")

stage_version = "week3-fyp-v1"
indexed_fingerprint = sha256(index_path.read_bytes()).hexdigest()
lineage = {
    "pipeline_version": stage_version,
    "private_manifest_sha256": manifest_sha256,
    "index": "data/processed/image_manifest.csv",
    "index_sha256": indexed_fingerprint,
    "record_count": len(prepared),
    "source_storage": "private smartpark_fyp_review_input.zip",
    "records": [
        {key: row[key] for key in ("record_id", "source_id", "sha256", "image_path", "annotation_origin")}
        for row in prepared
    ],
}
write_json(data / "week3_manifest.json", lineage)
write_json(pipeline / "pipeline_manifest.json", {
    "pipeline_version": stage_version,
    "data_type": "images and time-aligned replay metadata",
    "private_manifest_sha256": manifest_sha256,
    "prepared_index_sha256": indexed_fingerprint,
    "prepared_records": len(prepared),
    "by_role": dict(kind_counts),
    "capture_dates": sorted({row["capture_date"] for row in prepared if row["capture_date"]}),
    "replay_sessions": len({row["source_id"] for row in prepared if row["kind"] == "annotated_replay"}),
    "configuration": {"preserve_image_bytes": True, "format": "JPEG", "mode": "RGB",
                      "preserve_source_order": True},
    "repeatability": "PASS: same ordered rows, source fingerprints and mappings",
    "outputs": ["data/processed/image_manifest.csv", "data/week3_manifest.json"],
})
print(f"PASS: verified the same {len(second_run)} ordered rows, registrations and image hashes twice.")
print(f"Index SHA-256: {indexed_fingerprint}")
archive = output.with_name(output.name + "_handoff.zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as handoff:
    for path in (index_path, data / "week3_manifest.json", pipeline / "pipeline_manifest.json"):
        handoff.write(path, path.relative_to(output).as_posix())
print(f"Public hand-off ZIP: {archive.name}")

PASS: verified the same 285 ordered rows, registrations and image hashes twice.
Index SHA-256: 1967c41abb5e576891bb3c109ded4895fe44ab772470fcd5c21b8b87ef870b20
Public hand-off ZIP: fyp_canonical_handoff.zip


In [5]:
if COLAB:
    from google.colab import files

    files.download(str(archive))